# 24.11 SQL 面试速通 / SQL Interview Cram (真实可跑 / all runnable)

**中文**:**SQL 是数据科学面试的硬门槛**——几乎每一个数据/分析/DS 岗都会考,而且往往是第一轮筛人的关卡。好消息是:**SQL 面试题高度模式化**,来来回回就那几类经典套路。掌握了这几个模式(尤其**窗口函数**),LeetCode Hard 的 SQL 题也不过是它们的组合。本节用 **DuckDB(真实可跑的 SQL 引擎)** 现场演示数据科学面试最高频的 **7 大 SQL 模式**——全部是真实数据、真实查询、真实结果,你可以直接改数据验证。这是 Part 24 里最"手把手"的一课,也是最能直接提升面试通过率的一课。
**English**: **SQL is a hard gate in data-science interviews** — nearly every data/analytics/DS role tests it, often as the first screening round. The good news: **SQL interview questions are highly patterned**, cycling through a few classic templates. Master these patterns (especially **window functions**) and even LeetCode Hard SQL is just their combination. This section uses **DuckDB (a real, runnable SQL engine)** to demonstrate live the **7 most frequent SQL patterns** in data-science interviews — all real data, real queries, real results that you can edit and verify. This is Part 24's most hands-on lesson and the one most directly improving your interview pass rate.

---

**中文**:**SQL 面试的核心武器:窗口函数(window functions)**。一半以上的中高难度 SQL 面试题都靠它。核心语法:
**English**: **The core weapon of SQL interviews: window functions**. More than half of medium-to-hard SQL interview questions rely on them. Core syntax:
```sql
FUNCTION() OVER (PARTITION BY 分组列 ORDER BY 排序列 [ROWS BETWEEN ...])
```
- **中文**:**排名类**:`ROW_NUMBER()`(唯一序号)、`RANK()`(并列跳号)、`DENSE_RANK()`(并列不跳号)——做"每组 top-N"。
  **Ranking**: `ROW_NUMBER()` (unique sequence), `RANK()` (ties skip), `DENSE_RANK()` (ties don't skip) — for "top-N per group."
- **中文**:**偏移类**:`LAG()`(取前一行)、`LEAD()`(取后一行)——做"环比/同比、与上一条比较"。
  **Offset**: `LAG()` (previous row), `LEAD()` (next row) — for "period-over-period, compare with the previous row."
- **中文**:**聚合类**:`SUM()/AVG()/COUNT() OVER(...)`——做"累计和、移动平均、占比"(不折叠行)。
  **Aggregate**: `SUM()/AVG()/COUNT() OVER(...)` — for "running total, moving average, share of total" (without collapsing rows).

> 💡 **面试速查 / Interview cheat-sheet（★★★ SQL 面试, 逢面必考）**
> **中文**:**SQL 面试 7 大模式**:①**窗口函数排名**(ROW_NUMBER/RANK/DENSE_RANK 做每组top-N, 如第2高薪)②**LAG/LEAD**(环比增长、与上条比)③**累计聚合**(SUM OVER 做 running total/移动平均/占比)④**自连接 self-join**(员工vs经理、同表关联)⑤**漏斗/转化**(COUNT DISTINCT 分步 + 占比)⑥**gaps-and-islands**(连续段: 日期 − ROW_NUMBER 得到"岛"标识, 经典难题)⑦**留存/cohort**(首次活跃 + LEFT JOIN 后续活跃算 DN 留存)。**其他高频**:GROUP BY + HAVING(过滤聚合)、CASE WHEN(条件聚合/透视)、CTE(WITH, 拆解复杂查询, 可读)、日期函数(DATE_TRUNC/DATE_DIFF)、COALESCE(空值)、DISTINCT。**关键技巧**:①**用 CTE(WITH)分步**别写巨大嵌套②窗口函数不折叠行(vs GROUP BY 折叠)③**每组top-N = 窗口排名 + 外层过滤 rk<=N**④NULL 处理(COUNT 忽略 NULL、LEFT JOIN 找不匹配、COALESCE)⑤先想清楚粒度(一行代表什么)。**练习**:LeetCode SQL(Medium/Hard)、StrataScratch、DataLemur。面试金句:*"SQL 面试核心是窗口函数: 排名(ROW_NUMBER/DENSE_RANK 做每组top-N)、LAG/LEAD(环比)、SUM OVER(累计/占比); 加上自连接、漏斗、gaps-and-islands(日期减行号找连续段)、留存 cohort; 用 CTE 分步拆解复杂查询、注意 NULL 和粒度。"*
> **English**: **7 SQL interview patterns**: ① **window ranking** (ROW_NUMBER/RANK/DENSE_RANK for top-N per group, e.g. 2nd highest salary) ② **LAG/LEAD** (period-over-period growth, compare with previous row) ③ **cumulative aggregate** (SUM OVER for running total/moving average/share) ④ **self-join** (employee vs manager, same-table linkage) ⑤ **funnel/conversion** (COUNT DISTINCT per step + share) ⑥ **gaps-and-islands** (consecutive runs: date − ROW_NUMBER gives an "island" identifier, a classic hard problem) ⑦ **retention/cohort** (first activity + LEFT JOIN later activity for DN retention). **Other frequent**: GROUP BY + HAVING (filter aggregates), CASE WHEN (conditional aggregation/pivot), CTE (WITH, decompose complex queries, readable), date functions (DATE_TRUNC/DATE_DIFF), COALESCE (nulls), DISTINCT. **Key techniques**: ① **use CTEs (WITH) to step through**, don't write giant nesting ② window functions don't collapse rows (vs GROUP BY collapses) ③ **top-N per group = window ranking + outer filter rk<=N** ④ NULL handling (COUNT ignores NULL, LEFT JOIN finds non-matches, COALESCE) ⑤ think through granularity first (what one row represents). **Practice**: LeetCode SQL (Medium/Hard), StrataScratch, DataLemur. Interview line: *"SQL interviews center on window functions: ranking (ROW_NUMBER/DENSE_RANK for top-N per group), LAG/LEAD (period-over-period), SUM OVER (cumulative/share); plus self-joins, funnels, gaps-and-islands (date minus row number to find consecutive runs), retention cohorts; use CTEs to step through complex queries and mind NULLs and granularity."*


In [ ]:

# ============================================================
# 建表:员工、销售、事件、登录(真实数据)/ setup: employees, sales, events, logins (real data)
# 中文:用 DuckDB(真实 SQL 引擎)。下面每个模式都是真实查询+真实结果, 你可以改数据验证。
# English: using DuckDB (a real SQL engine). Each pattern below is a real query + real result you can edit and verify.
# ============================================================
import duckdb
con=duckdb.connect()
con.execute("""CREATE TABLE employees AS SELECT * FROM (VALUES
  (1,'Alice','Eng',120000,2),(2,'Bob','Eng',95000,2),(3,'Carol','Eng',105000,2),
  (4,'Dave','Sales',80000,5),(5,'Eve','Sales',110000,NULL),(6,'Frank','Sales',75000,5)
) t(id,name,dept,salary,mgr_id)""")
con.execute("""CREATE TABLE sales AS SELECT * FROM (VALUES
  ('2024-01',100),('2024-02',150),('2024-03',120),('2024-04',200)) t(month,revenue)""")
con.execute("""CREATE TABLE events AS SELECT * FROM (VALUES
  (1,'view'),(1,'cart'),(1,'purchase'),(2,'view'),(2,'cart'),(3,'view'),(4,'view'),(4,'cart'),(4,'purchase')
) t(user_id,step)""")
con.execute("""CREATE TABLE logins AS SELECT * FROM (VALUES
  (1,DATE '2024-01-01'),(1,DATE '2024-01-02'),(1,DATE '2024-01-03'),(1,DATE '2024-01-05'),(1,DATE '2024-01-06')
) t(user_id,d)""")
con.execute("""CREATE TABLE activity AS SELECT * FROM (VALUES
  (1,DATE '2024-01-01'),(1,DATE '2024-01-02'),(2,DATE '2024-01-01'),(3,DATE '2024-01-01'),(3,DATE '2024-01-02')
) t(user_id,d)""")
print("建表完成 / tables ready: employees, sales, events, logins, activity")


In [ ]:

# ============================================================
# 模式①②③:窗口函数(排名/偏移/累计)/ patterns ①②③: window functions (rank/offset/cumulative)
# ============================================================
print("① 每部门薪资第2高(DENSE_RANK + 外层过滤, 经典'每组top-N')/ 2nd highest salary per dept:")
print(con.sql("""
SELECT dept, name, salary FROM (
  SELECT *, DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rk   -- 每部门内按薪资排名
  FROM employees
) WHERE rk = 2                                                                  -- 取第2名
""").df().to_string(index=False))

print("\n② 月环比增长(LAG 取上月, 算增长率)/ month-over-month growth with LAG:")
print(con.sql("""
SELECT month, revenue,
  revenue - LAG(revenue) OVER (ORDER BY month) AS mom_change,                   -- 与上月的差
  ROUND(100.0*(revenue - LAG(revenue) OVER (ORDER BY month))
        / LAG(revenue) OVER (ORDER BY month), 1) AS mom_pct                     -- 环比增长率 %
FROM sales ORDER BY month
""").df().to_string(index=False))

print("\n③ 累计收入 running total(SUM OVER, 不折叠行)/ running total:")
print(con.sql("""
SELECT month, revenue,
  SUM(revenue) OVER (ORDER BY month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative             -- 从头累计到当前行
FROM sales ORDER BY month
""").df().to_string(index=False))


In [ ]:

# ============================================================
# 模式④⑤:自连接 + 漏斗 / patterns ④⑤: self-join + funnel
# ============================================================
print("④ 薪资高于其经理的员工(自连接: 同表按 mgr_id 关联)/ employees earning more than their manager:")
print(con.sql("""
SELECT e.name AS employee, e.salary AS emp_salary, m.name AS manager, m.salary AS mgr_salary
FROM employees e
JOIN employees m ON e.mgr_id = m.id       -- 员工表自己连自己(e=员工, m=经理)
WHERE e.salary > m.salary
""").df().to_string(index=False))

print("\n⑤ 漏斗转化率(每步去重用户数 + 相对首步占比)/ funnel conversion:")
print(con.sql("""
WITH step_users AS (
  SELECT step, COUNT(DISTINCT user_id) AS users FROM events GROUP BY step
)
SELECT step, users,
  ROUND(100.0*users / MAX(users) OVER (), 1) AS pct_of_top                      -- 相对最大步的占比
FROM step_users
ORDER BY CASE step WHEN 'view' THEN 1 WHEN 'cart' THEN 2 ELSE 3 END
""").df().to_string(index=False))


In [ ]:

# ============================================================
# 模式⑥⑦:gaps-and-islands + 留存 cohort(两大经典难题)/ patterns ⑥⑦: gaps-and-islands + retention
# ============================================================
print("⑥ 连续登录段 gaps-and-islands(经典难题: 日期 − 行号 → 同一连续段的差值相同)/ consecutive login streaks:")
print(con.sql("""
WITH grp AS (
  SELECT user_id, d,
    d - INTERVAL (ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d)) DAY AS island  -- 关键技巧: 连续日期减递增行号 = 常数
  FROM logins
)
SELECT user_id, MIN(d) AS streak_start, MAX(d) AS streak_end, COUNT(*) AS streak_len
FROM grp GROUP BY user_id, island ORDER BY streak_start
""").df().to_string(index=False))

print("\n⑦ 次日留存 D1 retention(首次活跃 + LEFT JOIN 次日是否活跃)/ Day-1 retention:")
print(con.sql("""
WITH first_day AS (SELECT user_id, MIN(d) AS fd FROM activity GROUP BY user_id)  -- 每个用户首次活跃日
SELECT ROUND(100.0*COUNT(DISTINCT a.user_id) / COUNT(DISTINCT f.user_id), 1) AS d1_retention_pct
FROM first_day f
LEFT JOIN activity a ON a.user_id=f.user_id AND a.d = f.fd + INTERVAL 1 DAY      -- 次日是否有活跃
""").df().to_string(index=False))
print("\n→ 7 大模式覆盖了数据科学 SQL 面试的绝大多数题型; 复杂题不过是它们的组合。用 CTE(WITH)分步是关键。")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **SQL 面试的高通过率来自"识别模式",而非"临场硬想"**:我们演示的 7 个模式——窗口排名、LAG/LEAD、累计聚合、自连接、漏斗、gaps-and-islands、留存——覆盖了数据科学 SQL 面试的绝大多数题型。当你把这些模式练成肌肉记忆,拿到一道新题时,你的第一反应不是"这题好难",而是"这是每组 top-N,用窗口排名 + 外层过滤"或"这是找连续段,用日期减行号"。**LeetCode Hard 的 SQL 题,几乎都是这几个基础模式的组合和嵌套。** 尤其**窗口函数是分水岭**:不会窗口函数,中高难度 SQL 题基本做不出;熟练窗口函数,一半的题迎刃而解。所以 SQL 备考的重点非常明确:**把窗口函数和这 7 个模式练熟**。
2. **两个最能区分水平的技巧:CTE 分步 + 理解粒度**:①**用 CTE(`WITH`)把复杂查询拆成可读的步骤**,而不是写一个五层嵌套的巨型子查询——这不仅让你自己不出错,也让面试官能跟上你的思路(面试时你要边写边讲)。我们的每个复杂查询(漏斗、gaps-and-islands、留存)都用了 CTE 分步,清晰可读。②**永远先想清楚"一行代表什么"(粒度 granularity)**:很多 SQL bug 源于 join 后粒度变了(一对多导致行翻倍、聚合错误)。写之前先问:这张表一行是一个用户?一个用户一天?一笔交易?搞清楚粒度,join 和聚合就不会错。③**NULL 是暗坑**:`COUNT(*)` 数所有行但 `COUNT(列)` 忽略 NULL;`LEFT JOIN` 后未匹配的是 NULL(留存题正是靠这个);`NULL = NULL` 不成立(要用 `IS NULL`);`COALESCE` 兜底。这些细节区分"会写 SQL"和"SQL 严谨"。
3. **诚实的边界:面试 SQL ≠ 生产 SQL,但基本功相通**。①**面试考的是逻辑能力(能不能用 SQL 表达一个复杂的数据问题),而非性能优化**——面试里你几乎不用担心索引、执行计划、大表性能(那是 Part 21 的内容);但真实工作里,同样的查询在十亿行表上就要考虑分区裁剪、避免全表扫描、减少 shuffle。基本功(窗口函数、join、聚合的正确性)是两者相通的地基。②**别只会背模式,要理解为什么**:比如 gaps-and-islands 里"日期减行号"为什么能标识连续段?因为连续的日期和连续的行号增速相同,相减得到常数;一旦断开,差值就变——理解了原理,变体题(连续登录、连续盈利月、连续排名)你都能推导,而不是死记一个模板。③**多方言但核心通用**:MySQL/PostgreSQL/DuckDB/Spark SQL/BigQuery 语法有细微差异(日期函数、字符串函数各家不同),但窗口函数、CTE、join 这些核心是通用的——学会一种,换方言只是查语法。④**刻意练习是唯一捷径**:SQL 是熟练度活儿,LeetCode(Medium/Hard)、StrataScratch、DataLemur 上刷够 50-100 题,把这 7 个模式练到条件反射,面试 SQL 关就稳了。**结论:SQL 面试高度模式化, 核心是窗口函数 + 7 大经典模式(排名/偏移/累计/自连接/漏斗/gaps-and-islands/留存), 复杂题是它们的组合; 用 CTE 分步拆解、先想清楚粒度、小心 NULL 是区分水平的关键; 面试考逻辑表达而非性能, 但基本功与生产相通——刻意刷题把模式练成肌肉记忆, 是通过 SQL 关最可靠的路。**

**English**:
1. **A high SQL-interview pass rate comes from "recognizing patterns," not "figuring it out on the spot"**: the 7 patterns we demonstrated — window ranking, LAG/LEAD, cumulative aggregation, self-join, funnel, gaps-and-islands, retention — cover the vast majority of data-science SQL interview questions. When you drill these to muscle memory, your first reaction to a new question isn't "this is hard" but "this is top-N per group, use window ranking + outer filter" or "this is finding consecutive runs, use date minus row number." **LeetCode Hard SQL is almost always a combination and nesting of these basic patterns.** Especially **window functions are the dividing line**: without them you basically can't do medium-to-hard SQL; fluent with them, half the questions fall easily. So SQL prep has a very clear focus: **master window functions and these 7 patterns.**
2. **Two skills that most distinguish level: CTE stepping + understanding granularity**: ① **use CTEs (`WITH`) to break complex queries into readable steps**, rather than a five-level nested giant subquery — this keeps you error-free and lets the interviewer follow your reasoning (you write and explain simultaneously in interviews). Each of our complex queries (funnel, gaps-and-islands, retention) uses CTE stepping, clear and readable. ② **Always think through "what one row represents" (granularity) first**: many SQL bugs come from granularity changing after a join (one-to-many doubling rows, wrong aggregation). Before writing, ask: is one row a user? a user-day? a transaction? Get the granularity right and joins and aggregations won't go wrong. ③ **NULL is a hidden trap**: `COUNT(*)` counts all rows but `COUNT(column)` ignores NULL; after `LEFT JOIN` non-matches are NULL (retention questions rely on exactly this); `NULL = NULL` doesn't hold (use `IS NULL`); `COALESCE` for fallback. These details distinguish "can write SQL" from "rigorous SQL."
3. **Honest limits: interview SQL ≠ production SQL, but the fundamentals overlap**. ① **Interviews test logical ability (can you express a complex data problem in SQL), not performance optimization** — in interviews you rarely worry about indexes, execution plans, big-table performance (that's Part 21); but in real work, the same query on a billion-row table requires partition pruning, avoiding full scans, reducing shuffle. The fundamentals (correctness of window functions, joins, aggregation) are the shared foundation. ② **Don't just memorize patterns, understand why**: e.g., why does "date minus row number" identify consecutive runs in gaps-and-islands? Because consecutive dates and consecutive row numbers increase at the same rate, so their difference is constant; once broken, the difference changes — understand the principle and you can derive variants (consecutive logins, consecutive profitable months, consecutive rankings), not memorize a template. ③ **Multiple dialects but a common core**: MySQL/PostgreSQL/DuckDB/Spark SQL/BigQuery differ slightly (date/string functions vary), but window functions, CTEs, joins are universal — learn one and switching dialects is just checking syntax. ④ **Deliberate practice is the only shortcut**: SQL is a fluency skill; grind 50-100 questions on LeetCode (Medium/Hard), StrataScratch, DataLemur, drilling these 7 patterns to reflex, and the SQL round is secure. **Conclusion: SQL interviews are highly patterned, centered on window functions + the 7 classic patterns (ranking/offset/cumulative/self-join/funnel/gaps-and-islands/retention), with complex questions being their combinations; using CTEs to step through, thinking through granularity first, and minding NULLs distinguish level; interviews test logical expression not performance, but the fundamentals overlap with production — deliberately grinding problems to drill patterns into muscle memory is the most reliable path through the SQL round."*

> 💼 **实战视角 / Practical angle**
> **中文**:SQL 备考:①**先吃透窗口函数**(排名 ROW_NUMBER/RANK/DENSE_RANK、偏移 LAG/LEAD、聚合 SUM/AVG OVER + frame ROWS BETWEEN);②**练熟 7 大模式**(本节的), 尤其 gaps-and-islands 和留存(高频难题);③**用 CTE 分步**写复杂查询, 别嵌套地狱;④**刷题**: LeetCode SQL 50 + StrataScratch/DataLemur, 覆盖各公司真题;⑤**面试时边写边讲**(说清粒度、思路、边界), 先写框架 CTE 再填;⑥**注意方言差异**(日期函数各家不同)但核心通用。真实工作:同样的模式 + 加上性能(分区/索引/避免全表扫)、可读性(CTE/注释)、正确性(NULL/去重/粒度)。面试金句:*"SQL 面试核心是窗口函数(排名做每组top-N、LAG/LEAD 做环比、SUM OVER 做累计)+ 自连接/漏斗/gaps-and-islands(日期减行号)/留存 cohort 这 7 大模式, 复杂题是组合; 用 CTE 分步、先想清粒度、小心 NULL; 刷 LeetCode/StrataScratch 把模式练成肌肉记忆。"*
> **English**: SQL prep: ① **master window functions first** (ranking ROW_NUMBER/RANK/DENSE_RANK, offset LAG/LEAD, aggregate SUM/AVG OVER + frame ROWS BETWEEN); ② **drill the 7 patterns** (this section's), especially gaps-and-islands and retention (frequent hard ones); ③ **use CTEs to step through** complex queries, avoid nesting hell; ④ **grind problems**: LeetCode SQL 50 + StrataScratch/DataLemur, covering companies' real questions; ⑤ **write and explain in interviews** (state granularity, reasoning, edge cases), sketch the CTE framework first then fill; ⑥ **mind dialect differences** (date functions vary) but the core is universal. Real work: the same patterns + performance (partition/index/avoid full scan), readability (CTE/comments), correctness (NULL/dedup/granularity). Interview line: *"SQL interviews center on window functions (ranking for top-N per group, LAG/LEAD for period-over-period, SUM OVER for cumulative) + the 7 patterns of self-join/funnel/gaps-and-islands (date minus row number)/retention cohort, with complex questions being combinations; use CTEs to step through, think through granularity first, mind NULLs; grind LeetCode/StrataScratch to drill patterns into muscle memory."*

---
### 小结 / Summary
- **中文**:SQL 面试高度模式化, 核心是窗口函数 + 7 大模式:①排名(每组top-N)②LAG/LEAD(环比)③累计聚合④自连接⑤漏斗⑥gaps-and-islands⑦留存 cohort。
- **English**: SQL interviews are highly patterned, centered on window functions + 7 patterns: ① ranking (top-N per group) ② LAG/LEAD (period-over-period) ③ cumulative aggregate ④ self-join ⑤ funnel ⑥ gaps-and-islands ⑦ retention cohort.
- **中文**:关键技巧:CTE(WITH)分步拆解、先想清粒度、小心 NULL; 复杂题是基础模式的组合。
- **English**: Key techniques: CTE (WITH) stepping, think through granularity first, mind NULLs; complex questions are combinations of basic patterns.
- **中文**:面试考逻辑表达而非性能(性能见 Part 21); 刻意刷题(LeetCode/StrataScratch)把模式练成肌肉记忆是通关最可靠的路。
- **English**: Interviews test logical expression not performance (performance in Part 21); deliberately grinding problems (LeetCode/StrataScratch) to drill patterns into muscle memory is the most reliable path.
